# Stage 3 — Baseline Diarization

Append these cells to the notebook that already has your inputs set up.

**Assumes already set up by your existing cells:** `AUDIO`, `/kaggle/working/data/ref/`,
the scripts copied into `/kaggle/working/`, and `HF_TOKEN` in the environment.

The scripts come from the code dataset, so **re-upload `sarvam-diar-code` whenever
a script changes** — otherwise Kaggle runs a stale copy.

**Metric policy (deliberately unforgiving):** `collar=0.0`, `skip_overlap=False`,
UEM = the full clip. Overlapping speech **is** scored, and boundary errors get no
forgiveness. Expect DER well above published numbers — those almost always use
`collar=0.25` and skip overlap.

Corpus: 99 clips / 12.28 h / 9 Indic languages / 7.60% overlapped speech.

In [ ]:
# sanity check: the things the cells below depend on
print("AUDIO :", AUDIO, f"({len(list(AUDIO.glob('*.wav')))} wavs)")
import os, pathlib
print("ref   :", pathlib.Path("/kaggle/working/data/ref/clip_meta.csv").exists())
print("token :", bool(os.environ.get("HF_TOKEN")))
print("scripts:", sorted(p.name for p in pathlib.Path("/kaggle/working").glob("stage*.py")))

## Smoke test (5 clips)

Check before committing ~25 min of GPU:
- `device=cuda` — if it says `cpu`, the accelerator is off
- **RTF ≈ 0.03** (~30× realtime). Near 1.0 means it is silently on CPU
- `pyannote.audio` version — 4.x means `community1` is available as System B

In [ ]:
!python stage3_diarize.py --system pyannote31 --data data --wav-dir {AUDIO} --limit 5

## Full run — pyannote 3.1

Resumable: re-run this cell after a session death and it skips completed clips.

In [ ]:
!python stage3_diarize.py --system pyannote31 --data data --wav-dir {AUDIO}

In [ ]:
!python stage3_score.py --data data --systems pyannote31 --diagnostic

## System B

Two options — try Sortformer first, fall back to community-1 if NeMo fights the
preinstalled torch.

**Sortformer is hard-capped at 4 speakers**, and 17 of our clips have ≥5. It
cannot represent those, so report its per-speaker-count breakdown rather than
only the headline DER — the cap is a model limitation, not a quality result.

In [ ]:
# Option B1: NeMo Sortformer. If this breaks torch, restart and use B2.
!pip install -q "nemo_toolkit[asr]"

In [ ]:
!python stage3_diarize.py --system sortformer --data data --wav-dir {AUDIO} --limit 5

In [ ]:
!python stage3_diarize.py --system sortformer --data data --wav-dir {AUDIO}

### Sortformer streaming — required for the long clips

Offline Sortformer attends over the whole session, so peak VRAM grows as
**O(duration²)**: a 913 s clip asked for 7.8 GiB, an 1822 s clip for 30.9 GiB.
On a 14.6 GiB T4 that OOM'd on 25 of 99 clips — and those 25 are systematically
the *longest* clips, which in this corpus also carry the most speakers and
overlap. Scoring the surviving 74 against pyannote's 99 would compare two
different corpora.

`sortformer_stream` is the same checkpoint with `streaming_mode = True`: fixed
chunks, with a speaker cache + FIFO queue carrying identity across boundaries,
so no stitching is needed on our side. It writes to its own `hyp/` directory,
so the offline hypotheses survive for the streaming-vs-offline comparison on the
74 clips where both ran.

In [ ]:
!python stage3_diarize.py --system sortformer_stream --data data --wav-dir {AUDIO} --limit 5

In [ ]:
!python stage3_diarize.py --system sortformer_stream --data data --wav-dir {AUDIO}

### Option B2 — pyannote community-1 (fallback, needs pyannote.audio ≥ 4)

In [ ]:
# !python stage3_diarize.py --system community1 --data data --wav-dir {AUDIO} --limit 5
# !python stage3_diarize.py --system community1 --data data --wav-dir {AUDIO}

## Score everything

In [ ]:
!python stage3_score.py --data data --systems pyannote31 sortformer sortformer_stream --diagnostic

In [ ]:
import pandas as pd
pd.read_csv("/kaggle/working/data/results/diarization_summary.csv")

## Save

**Save Version → Quick Save** before the session ends. `/kaggle/working` is wiped
on timeout, and the hypothesis RTTMs represent the GPU time you just spent.

Next session: attach this notebook's output as a data source and the runs resume
from their manifests.

In [ ]:
!du -sh /kaggle/working/data/* 2>/dev/null
!find /kaggle/working/data/hyp -name "*.rttm" | wc -l